In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


In [2]:
!pip install -q "datasets<4.0" sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 7.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 3.9 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pathos 0.3.5 requires dill>=0.4.1, but you have dill 0.3.8 which is incompatible.
pathos 0.3.5 requires multiprocess>=0.70.19, but you have multiprocess 0.70.16 which is incompatible.
s3fs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
gcsfs 2025.12.0 requires fsspec==20

In [3]:
import datasets, sentence_transformers, torch
print(datasets.__version__, sentence_transformers.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

3.6.0 5.7.0
True
Tesla T4


In [25]:
import pandas as pd
from datasets import load_dataset

claims = load_dataset("allenai/scifact", "claims", trust_remote_code=True)
corpus = load_dataset("allenai/scifact", "corpus", trust_remote_code=True)

val = pd.DataFrame(claims["validation"])
sent_map = {int(r["doc_id"]): list(r["abstract"]) for r in corpus["train"]}

print(len(val), "validation rows")
print(len(sent_map), "documents")

450 validation rows
5183 documents


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = "cuda"
name = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"

tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForSequenceClassification.from_pretrained(name).to(device).eval()

id2label = model.config.id2label
label_map = {"entailment": "SUPPORT", "contradiction": "CONTRADICT", "neutral": "NOINFO"}
print(id2label)

config.json:   0%|          | 0.00/1.09k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  369MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

{0: 'entailment', 1: 'neutral', 2: 'contradiction'}


In [6]:
import time
from sklearn.metrics import classification_report, confusion_matrix

def pick_doc(r):
    ev = str(r["evidence_doc_id"]).strip()
    return int(ev) if ev != "" else int(r["cited_doc_ids"][0])

# Evaluation table banao
rows = []
for _, r in val.iterrows():
    d = pick_doc(r)
    if d in sent_map:
        rows.append((r["id"], r["claim"], d, r["evidence_label"] or "NOINFO"))
df = pd.DataFrame(rows, columns=["id", "claim", "doc_id", "gold"])
print(len(df), "pairs")

# Batch mein prediction
def run(pairs, bs=16):
    preds = []
    for i in range(0, len(pairs), bs):
        b = pairs[i:i+bs]
        enc = tok([p[0] for p in b], [p[1] for p in b],
                  truncation="only_first", max_length=512,
                  padding=True, return_tensors="pt").to(device)
        with torch.no_grad():
            out = model(**enc).logits.argmax(-1).tolist()
        preds += [label_map[id2label[x]] for x in out]
    return preds

t0 = time.time()
df["pred_v1"] = run([(" ".join(sent_map[d]), c) for c, d in zip(df.claim, df.doc_id)])
print("time:", round(time.time() - t0, 1), "sec")

labels = ["SUPPORT", "CONTRADICT", "NOINFO"]
print(classification_report(df.gold, df.pred_v1, digits=3))
print(confusion_matrix(df.gold, df.pred_v1, labels=labels))

450 pairs
time: 10.1 sec
              precision    recall  f1-score   support

  CONTRADICT      0.667     0.590     0.626       122
      NOINFO      0.554     0.777     0.647       112
     SUPPORT      0.778     0.667     0.718       216

    accuracy                          0.673       450
   macro avg      0.666     0.678     0.664       450
weighted avg      0.692     0.673     0.675       450

[[144  25  47]
 [ 27  72  23]
 [ 14  11  87]]


In [7]:
from sentence_transformers import SentenceTransformer, util

emb = SentenceTransformer("all-MiniLM-L6-v2", device=device)

def top_sents(claim, sents, k=3):
    c = emb.encode(claim, convert_to_tensor=True)
    s = emb.encode(sents, convert_to_tensor=True)
    idx = util.cos_sim(c, s)[0].topk(min(k, len(sents))).indices.tolist()
    return " ".join(sents[i] for i in sorted(idx))

t0 = time.time()
df["pred_v2"] = run([(top_sents(c, sent_map[d]), c) for c, d in zip(df.claim, df.doc_id)])
print("time:", round(time.time() - t0, 1), "sec")

print(classification_report(df.gold, df.pred_v2, digits=3))
print(confusion_matrix(df.gold, df.pred_v2, labels=labels))

df.to_csv("val_predictions_v1_v2.csv", index=False)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

time: 12.8 sec
              precision    recall  f1-score   support

  CONTRADICT      0.656     0.656     0.656       122
      NOINFO      0.468     0.795     0.589       112
     SUPPORT      0.848     0.542     0.661       216

    accuracy                          0.636       450
   macro avg      0.657     0.664     0.635       450
weighted avg      0.701     0.636     0.642       450

[[117  27  72]
 [ 13  80  29]
 [  8  15  89]]


In [8]:
ev_rows = []
for _, r in val.iterrows():
    d = pick_doc(r)
    lab = r["evidence_label"]
    if d in sent_map and lab in ("SUPPORT", "CONTRADICT") and len(r["evidence_sentences"]) > 0:
        sents = sent_map[d]
        gold_idx = [i for i in r["evidence_sentences"] if i < len(sents)]
        if gold_idx:
            ev_rows.append((r["claim"], d, lab, sorted(gold_idx)))
ev = pd.DataFrame(ev_rows, columns=["claim", "doc_id", "gold", "gold_idx"])
print(len(ev), "claims with gold evidence")

# kya top-3 mein koi gold sentence aaya?
def top_idx(claim, sents, k=3):
    c = emb.encode(claim, convert_to_tensor=True)
    s = emb.encode(sents, convert_to_tensor=True)
    return util.cos_sim(c, s)[0].topk(min(k, len(sents))).indices.tolist()

hit = [len(set(top_idx(c, sent_map[d])) & set(g)) > 0
       for c, d, g in zip(ev.claim, ev.doc_id, ev.gold_idx)]
print("top-3 mein gold sentence aaya:", round(sum(hit) / len(hit), 3))

# 3 tarike se verify
ev["pred_full"] = run([(" ".join(sent_map[d]), c) for c, d in zip(ev.claim, ev.doc_id)])
ev["pred_top3"] = run([(top_sents(c, sent_map[d]), c) for c, d in zip(ev.claim, ev.doc_id)])
ev["pred_gold"] = run([(" ".join(sent_map[d][i] for i in g), c)
                       for c, d, g in zip(ev.claim, ev.doc_id, ev.gold_idx)])

for col in ["pred_full", "pred_top3", "pred_gold"]:
    print("=====", col)
    print(classification_report(ev.gold, ev[col], digits=3))

338 claims with gold evidence
top-3 mein gold sentence aaya: 0.627
===== pred_full
              precision    recall  f1-score   support

  CONTRADICT      0.742     0.590     0.658       122
      NOINFO      0.000     0.000     0.000         0
     SUPPORT      0.842     0.667     0.744       216

    accuracy                          0.639       338
   macro avg      0.528     0.419     0.467       338
weighted avg      0.806     0.639     0.713       338

===== pred_top3
              precision    recall  f1-score   support

  CONTRADICT      0.748     0.656     0.699       122
      NOINFO      0.000     0.000     0.000         0
     SUPPORT      0.900     0.542     0.676       216

    accuracy                          0.583       338
   macro avg      0.549     0.399     0.458       338
weighted avg      0.845     0.583     0.684       338

===== pred_gold
              precision    recall  f1-score   support

  CONTRADICT      0.870     0.656     0.748       122
      NOINFO  

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.p

In [9]:
!pip install -q rank_bm25
from rank_bm25 import BM25Okapi
import numpy as np

def tokens(t): return t.lower().split()

def rank_bm25(claim, sents):
    scores = BM25Okapi([tokens(s) for s in sents]).get_scores(tokens(claim))
    return list(np.argsort(-scores))

def rank_dense(claim, sents):
    c = emb.encode(claim, convert_to_tensor=True)
    s = emb.encode(sents, convert_to_tensor=True)
    return util.cos_sim(c, s)[0].argsort(descending=True).tolist()

def hit_at_k(rank_fn, k):
    hits = []
    for c, d, g in zip(ev.claim, ev.doc_id, ev.gold_idx):
        top = rank_fn(c, sent_map[d])[:k]
        hits.append(len(set(top) & set(g)) > 0)
    return round(float(np.mean(hits)), 3)

for k in [1, 3, 5]:
    print("k =", k, "| BM25:", hit_at_k(rank_bm25, k), "| MiniLM:", hit_at_k(rank_dense, k))

k = 1 | BM25: 0.29 | MiniLM: 0.343
k = 3 | BM25: 0.571 | MiniLM: 0.627
k = 5 | BM25: 0.74 | MiniLM: 0.805


In [10]:
from sklearn.metrics import confusion_matrix

def rank_rrf(claim, sents, k=60):
    scores = {}
    for ranking in (rank_bm25(claim, sents), rank_dense(claim, sents)):
        for pos, idx in enumerate(ranking):
            scores[idx] = scores.get(idx, 0) + 1 / (k + pos + 1)
    return sorted(scores, key=scores.get, reverse=True)

for k in [1, 3, 5]:
    print("k =", k, "| BM25:", hit_at_k(rank_bm25, k),
          "| MiniLM:", hit_at_k(rank_dense, k),
          "| Hybrid (RRF):", hit_at_k(rank_rrf, k))

print(confusion_matrix(ev.gold, ev.pred_gold, labels=["SUPPORT", "CONTRADICT", "NOINFO"]))

k = 1 | BM25: 0.29 | MiniLM: 0.343 | Hybrid (RRF): 0.32
k = 3 | BM25: 0.571 | MiniLM: 0.627 | Hybrid (RRF): 0.615
k = 5 | BM25: 0.74 | MiniLM: 0.805 | Hybrid (RRF): 0.787
[[ 77  12 127]
 [  4  80  38]
 [  0   0   0]]


In [26]:
train = pd.DataFrame(claims["train"])
tr_rows = []
for _, r in train.iterrows():
    d = pick_doc(r)
    if d not in sent_map:
        continue
    sents = sent_map[d]
    lab = r["evidence_label"] or "NOINFO"
    gi = [i for i in r["evidence_sentences"] if i < len(sents)]
    if lab in ("SUPPORT", "CONTRADICT") and gi:
        text = " ".join(sents[i] for i in sorted(gi))
    elif lab == "NOINFO":
        text = top_sents(r["claim"], sents)
    else:
        continue
    tr_rows.append((r["claim"], text, lab))
tr = pd.DataFrame(tr_rows, columns=["claim", "text", "label"])
print(len(tr))
print(tr.label.value_counts())
print(tr.sample(3, random_state=0).to_string())

1261
label
SUPPORT       616
CONTRADICT    341
NOINFO        304
Name: count, dtype: int64
                                                                                                                   claim                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     text    label
5    4-PBA treatment decreases endoplasmic reticulum stress in response to general endoplasmic reticulum stress markers.                                                                   Endoplasmic reticulum (ER) stress causes pancreatic β-cell dysfunction and contributes 

In [12]:
def build_text(claim, sents, gold_idx, k=3):
    ranked = rank_dense(claim, sents)
    chosen = list(gold_idx)
    for i in ranked:
        if len(chosen) >= min(k, len(sents)):
            break
        if i not in chosen:
            chosen.append(i)
    return " ".join(sents[i] for i in sorted(chosen))

tr_rows = []
for _, r in train.iterrows():
    d = pick_doc(r)
    if d not in sent_map:
        continue
    sents = sent_map[d]
    lab = r["evidence_label"] or "NOINFO"
    gi = [i for i in r["evidence_sentences"] if i < len(sents)]
    if lab in ("SUPPORT", "CONTRADICT") and gi:
        tr_rows.append((r["claim"], build_text(r["claim"], sents, gi), lab))
    elif lab == "NOINFO":
        tr_rows.append((r["claim"], top_sents(r["claim"], sents), lab))
tr = pd.DataFrame(tr_rows, columns=["claim", "text", "label"])
print(len(tr), tr.label.value_counts().to_dict())

1261 {'SUPPORT': 616, 'CONTRADICT': 341, 'NOINFO': 304}


In [28]:
import random
from torch.optim import AdamW

random.seed(0); torch.manual_seed(0)
lab2id = {"SUPPORT": 0, "NOINFO": 1, "CONTRADICT": 2}   # model ke order se match: entailment, neutral, contradiction
data = list(zip(tr.text, tr.claim, tr.label.map(lab2id)))

opt = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
EPOCHS, BS = 3, 16

model.train()
for ep in range(EPOCHS):
    random.shuffle(data)
    total = 0
    for i in range(0, len(data), BS):
        b = data[i:i+BS]
        enc = tok([x[0] for x in b], [x[1] for x in b], truncation="only_first",
                  max_length=256, padding=True, return_tensors="pt").to(device)
        y = torch.tensor([x[2] for x in b]).to(device)
        with torch.autocast("cuda", dtype=torch.float16):
            loss = model(**enc, labels=y).loss
        loss.backward()
        opt.step(); opt.zero_grad()
        total += loss.item()
    print("epoch", ep + 1, "avg loss", round(total / (len(data) / BS), 4))
model.eval()

# Validation: wahi 450 pairs, RETRIEVED top-3 sentences (v2 jaisa input)
df["pred_ft"] = run([(top_sents(c, sent_map[d]), c) for c, d in zip(df.claim, df.doc_id)])
print(classification_report(df.gold, df.pred_ft, digits=3))
print(confusion_matrix(df.gold, df.pred_ft, labels=labels))

epoch 1 avg loss nan
epoch 2 avg loss nan
epoch 3 avg loss nan
              precision    recall  f1-score   support

  CONTRADICT      0.000     0.000     0.000       122
      NOINFO      0.000     0.000     0.000       112
     SUPPORT      0.480     1.000     0.649       216

    accuracy                          0.480       450
   macro avg      0.160     0.333     0.216       450
weighted avg      0.230     0.480     0.311       450

[[216   0   0]
 [122   0   0]
 [112   0   0]]


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [27]:
print(torch.cuda.is_available(), len(tr), next(model.parameters()).device)

True 1261 cuda:0


In [30]:
import random, time
from torch.optim import AdamW

random.seed(0); torch.manual_seed(0)
lab2id = {"SUPPORT": 0, "NOINFO": 1, "CONTRADICT": 2}
data = list(zip(tr.text, tr.claim, tr.label.map(lab2id)))

opt = AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)
EPOCHS, BS = 3, 16

model.train()
t0 = time.time()
for ep in range(EPOCHS):
    random.shuffle(data)
    total, n = 0, 0
    for step, i in enumerate(range(0, len(data), BS)):
        b = data[i:i+BS]
        enc = tok([x[0] for x in b], [x[1] for x in b], truncation="only_first",
                  max_length=256, padding=True, return_tensors="pt").to(device)
        y = torch.tensor([x[2] for x in b]).to(device)
        loss = model(**enc, labels=y).loss
        if torch.isnan(loss):
            raise RuntimeError(f"nan loss at epoch {ep+1} step {step}")
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); opt.zero_grad()
        total += loss.item(); n += 1
        if step % 20 == 0:
            print(f"epoch {ep+1} step {step}/{len(data)//BS} loss {loss.item():.3f} | {time.time()-t0:.0f}s", flush=True)
    print("=== epoch", ep + 1, "avg loss", round(total / n, 4))
model.eval()

RuntimeError: nan loss at epoch 1 step 0

In [31]:
import torch
from transformers import AutoModelForSequenceClassification

name = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"
model = AutoModelForSequenceClassification.from_pretrained(name, torch_dtype=torch.float32).to(device)

lab2id = {"SUPPORT": 0, "NOINFO": 1, "CONTRADICT": 2}
b = list(zip(tr.text, tr.claim, tr.label.map(lab2id)))[:16]
enc = tok([x[0] for x in b], [x[1] for x in b], truncation="only_first",
          max_length=256, padding=True, return_tensors="pt").to(device)
y = torch.tensor([x[2] for x in b]).to(device)

model.eval()
with torch.no_grad():
    print("loss:", model(**enc, labels=y).loss.item())
print("any nan in weights:", any(torch.isnan(p).any().item() for p in model.parameters()))

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

loss: 1.421807050704956
any nan in weights: False


In [35]:
import random, time
from torch.optim import AdamW

random.seed(0); torch.manual_seed(0)
data = list(zip(tr.text, tr.claim, tr.label.map(lab2id)))

opt = AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)
EPOCHS, BS = 3, 16

model.train()
t0 = time.time()
for ep in range(EPOCHS):
    random.shuffle(data)
    total, n = 0, 0
    for step, i in enumerate(range(0, len(data), BS)):
        b = data[i:i+BS]
        enc = tok([x[0] for x in b], [x[1] for x in b], truncation="only_first",
                  max_length=256, padding=True, return_tensors="pt").to(device)
        y = torch.tensor([x[2] for x in b]).to(device)
        loss = model(**enc, labels=y).loss
        if torch.isnan(loss):
            raise RuntimeError(f"nan loss at epoch {ep+1} step {step}")
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); opt.zero_grad()
        total += loss.item(); n += 1
        if step % 20 == 0:
            print(f"epoch {ep+1} step {step}/{len(data)//BS} loss {loss.item():.3f} | {time.time()-t0:.0f}s", flush=True)
    print("=== epoch", ep + 1, "avg loss", round(total / n, 4))
model.eval()

epoch 1 step 0/78 loss 0.000 | 1s
epoch 1 step 20/78 loss 0.000 | 14s
epoch 1 step 40/78 loss 0.427 | 27s
epoch 1 step 60/78 loss 0.000 | 40s
=== epoch 1 avg loss 0.0073
epoch 2 step 0/78 loss 0.000 | 52s
epoch 2 step 20/78 loss 0.000 | 65s
epoch 2 step 40/78 loss 0.000 | 79s
epoch 2 step 60/78 loss 0.000 | 92s
=== epoch 2 avg loss 0.0064
epoch 3 step 0/78 loss 0.000 | 104s
epoch 3 step 20/78 loss 0.000 | 117s
epoch 3 step 40/78 loss 0.000 | 130s
epoch 3 step 60/78 loss 0.000 | 143s
=== epoch 3 avg loss 0.0055


DebertaV2ForSequenceClassification(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128100, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-11): 12 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): Layer

In [36]:
df["pred_ft"] = run([(top_sents(c, sent_map[d]), c) for c, d in zip(df.claim, df.doc_id)])
print(classification_report(df.gold, df.pred_ft, digits=3, zero_division=0))
print(confusion_matrix(df.gold, df.pred_ft, labels=labels))

              precision    recall  f1-score   support

  CONTRADICT      0.925     0.508     0.656       122
      NOINFO      0.441     0.938     0.600       112
     SUPPORT      0.945     0.634     0.759       216

    accuracy                          0.676       450
   macro avg      0.770     0.693     0.672       450
weighted avg      0.814     0.676     0.692       450

[[137   1  78]
 [  5  62  55]
 [  3   4 105]]


In [37]:
ev["pred_ft"] = run([(top_sents(c, sent_map[d]), c) for c, d in zip(ev.claim, ev.doc_id)])
ev["hit"] = hit
for h in [True, False]:
    g = ev[ev.hit == h]
    print("gold in top-3:", h, "| n =", len(g),
          "| accuracy:", round((g.gold == g.pred_ft).mean(), 3),
          "| predicted NOINFO:", round((g.pred_ft == "NOINFO").mean(), 3))

gold in top-3: True | n = 212 | accuracy: 0.599 | predicted NOINFO: 0.382
gold in top-3: False | n = 126 | accuracy: 0.571 | predicted NOINFO: 0.413


In [38]:
# A) Fine-tuned model ko gold evidence dene par kya hota hai?
ev["pred_ft_gold"] = run([(" ".join(sent_map[d][i] for i in g), c)
                          for c, d, g in zip(ev.claim, ev.doc_id, ev.gold_idx)])
print("gold input -> accuracy:", round((ev.gold == ev.pred_ft_gold).mean(), 3),
      "| NOINFO rate:", round((ev.pred_ft_gold == "NOINFO").mean(), 3))

# B) 10 galat examples jahan gold sentence top-3 mein thi par model ne NOINFO bola
bad = ev[(ev.hit) & (ev.pred_ft == "NOINFO")].head(10)
for _, r in bad.iterrows():
    print("\nCLAIM:", r.claim, "| GOLD:", r.gold)
    print("EVIDENCE:", " ".join(sent_map[r.doc_id][i] for i in r.gold_idx))

gold input -> accuracy: 0.917 | NOINFO rate: 0.012

CLAIM: 1,000 genomes project enables mapping of genetic sequence variation consisting of rare variants with larger penetrance effects than common variants. | GOLD: SUPPORT
EVIDENCE: We propose as an alternative explanation that variants much less common than the associated one may create "synthetic associations" by occurring, stochastically, more often in association with one of the alleles at the common site versus the other allele. We show that they are not only possible, but inevitable, and that under simple but reasonable genetic models, they are likely to account for or contribute to many of the recently identified signals reported in genome-wide association studies.

CLAIM: 1,000 genomes project enables mapping of genetic sequence variation consisting of rare variants with larger penetrance effects than common variants. | GOLD: SUPPORT
EVIDENCE: In conclusion, uncommon or rare genetic variants can easily create synthetic associa

In [39]:
import random, numpy as np
random.seed(0)

def make_input(c, d, g, n_extra, mode):
    sents = sent_map[d]; gold = sorted(g)
    if mode == "dense":
        extra = [i for i in rank_dense(c, sents) if i not in gold][:n_extra]
    else:
        pool = [i for i in range(len(sents)) if i not in gold]
        extra = random.sample(pool, min(n_extra, len(pool)))
    return " ".join(sents[i] for i in sorted(gold + extra))

for mode in ["dense", "random"]:
    for n in [0, 1, 2]:
        if mode == "random" and n == 0: continue
        preds = run([(make_input(c, d, g, n, mode), c)
                     for c, d, g in zip(ev.claim, ev.doc_id, ev.gold_idx)])
        print(mode, "extra =", n,
              "| acc:", round((ev.gold.values == np.array(preds)).mean(), 3),
              "| NOINFO rate:", round(np.mean([p == "NOINFO" for p in preds]), 3))

# kitne claims mein SAARE gold sentences top-3 mein aaye
full = [set(g) <= set(top_idx(c, sent_map[d])) for c, d, g in zip(ev.claim, ev.doc_id, ev.gold_idx)]
print("all gold in top-3:", round(np.mean(full), 3))
print(ev.gold_idx.str.len().value_counts().to_dict())

# leakage check
print(len(set(train.claim) & set(val.claim)), "claims overlap between train and val")

KeyboardInterrupt: 

In [40]:
import random, numpy as np
random.seed(0)

# har claim ka dense ranking ek baar nikalo
rank_cache = [rank_dense(c, sent_map[d]) for c, d in zip(ev.claim, ev.doc_id)]

def make_input(k, n_extra, mode):
    c, d, g = ev.claim.iloc[k], ev.doc_id.iloc[k], sorted(ev.gold_idx.iloc[k])
    sents = sent_map[d]
    if mode == "dense":
        extra = [i for i in rank_cache[k] if i not in g][:n_extra]
    else:
        pool = [i for i in range(len(sents)) if i not in g]
        extra = random.sample(pool, min(n_extra, len(pool)))
    return " ".join(sents[i] for i in sorted(g + extra))

for mode in ["dense", "random"]:
    for n in [0, 1, 2]:
        if mode == "random" and n == 0:
            continue
        preds = run([(make_input(k, n, mode), ev.claim.iloc[k]) for k in range(len(ev))])
        print(mode, "extra =", n,
              "| acc:", round((ev.gold.values == np.array(preds)).mean(), 3),
              "| NOINFO rate:", round(np.mean([p == "NOINFO" for p in preds]), 3))

# kitne claims mein SAARE gold sentences top-3 mein aaye
full = [set(g) <= set(r[:3]) for g, r in zip(ev.gold_idx, rank_cache)]
print("all gold in top-3:", round(np.mean(full), 3))
print(ev.gold_idx.str.len().value_counts().to_dict())

# leakage check
print(len(set(train.claim) & set(val.claim)), "claims overlap between train and val")

dense extra = 0 | acc: 0.917 | NOINFO rate: 0.012
dense extra = 1 | acc: 0.787 | NOINFO rate: 0.16
dense extra = 2 | acc: 0.586 | NOINFO rate: 0.388
random extra = 1 | acc: 0.805 | NOINFO rate: 0.115
random extra = 2 | acc: 0.615 | NOINFO rate: 0.34
all gold in top-3: 0.586
{1: 313, 2: 22, 3: 3}
2 claims overlap between train and val


In [41]:
from sentence_transformers import CrossEncoder
import numpy as np

ce = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=device)

def rank_ce(claim, sents):
    scores = ce.predict([(claim, s) for s in sents], show_progress_bar=False)
    return list(np.argsort(-np.array(scores)))

ce_rank = [rank_ce(c, sent_map[d]) for c, d in zip(ev.claim, ev.doc_id)]

for k in [1, 2, 3]:
    ce_hit = np.mean([len(set(r[:k]) & set(g)) > 0 for r, g in zip(ce_rank, ev.gold_idx)])
    de_hit = np.mean([len(set(r[:k]) & set(g)) > 0 for r, g in zip(rank_cache, ev.gold_idx)])
    print("k =", k, "| MiniLM bi-encoder:", round(de_hit, 3), "| cross-encoder:", round(ce_hit, 3))

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

k = 1 | MiniLM bi-encoder: 0.343 | cross-encoder: 0.402
k = 2 | MiniLM bi-encoder: 0.503 | cross-encoder: 0.586
k = 3 | MiniLM bi-encoder: 0.627 | cross-encoder: 0.698


In [42]:
# poore 450 pairs ke liye cross-encoder ranking (ek baar, cache)
df_rank = [rank_ce(c, sent_map[d]) for c, d in zip(df.claim, df.doc_id)]

def ce_input(k_idx, topk):
    sents = sent_map[df.doc_id.iloc[k_idx]]
    return " ".join(sents[i] for i in sorted(df_rank[k_idx][:topk]))

for topk in [1, 2, 3]:
    p = run([(ce_input(i, topk), df.claim.iloc[i]) for i in range(len(df))])
    print("top-k =", topk, "| acc:", round((df.gold.values == np.array(p)).mean(), 3))
    print(classification_report(df.gold, p, digits=3, zero_division=0))

top-k = 1 | acc: 0.691
              precision    recall  f1-score   support

  CONTRADICT      0.689     0.746     0.717       122
      NOINFO      0.596     0.304     0.402       112
     SUPPORT      0.713     0.861     0.780       216

    accuracy                          0.691       450
   macro avg      0.666     0.637     0.633       450
weighted avg      0.677     0.691     0.669       450

top-k = 2 | acc: 0.733
              precision    recall  f1-score   support

  CONTRADICT      0.754     0.705     0.729       122
      NOINFO      0.543     0.679     0.603       112
     SUPPORT      0.857     0.778     0.816       216

    accuracy                          0.733       450
   macro avg      0.718     0.720     0.716       450
weighted avg      0.751     0.733     0.739       450

top-k = 3 | acc: 0.676
              precision    recall  f1-score   support

  CONTRADICT      0.882     0.549     0.677       122
      NOINFO      0.456     0.920     0.609       112
     S

In [43]:
tr = tr[~tr.claim.isin(set(val.claim))].reset_index(drop=True)
print(len(tr))

1259


In [44]:
from transformers import AutoModelForSequenceClassification
import torch

name = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"
model = AutoModelForSequenceClassification.from_pretrained(name, dtype=torch.float32).to(device)
model.eval()

lab2id = {"SUPPORT": 0, "NOINFO": 1, "CONTRADICT": 2}
b = list(zip(tr.text, tr.claim, tr.label.map(lab2id)))[:16]
enc = tok([x[0] for x in b], [x[1] for x in b], truncation="only_first",
          max_length=256, padding=True, return_tensors="pt").to(device)
y = torch.tensor([x[2] for x in b]).to(device)
with torch.no_grad():
    print("fresh model loss on first 16:", round(model(**enc, labels=y).loss.item(), 3))

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

fresh model loss on first 16: 1.422


In [45]:
import random, time
from torch.optim import AdamW

random.seed(0); torch.manual_seed(0)
data = list(zip(tr.text, tr.claim, tr.label.map(lab2id)))

opt = AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)
EPOCHS, BS = 3, 16

model.train()
t0 = time.time()
for ep in range(EPOCHS):
    random.shuffle(data)
    total, n = 0, 0
    for step, i in enumerate(range(0, len(data), BS)):
        b = data[i:i+BS]
        enc = tok([x[0] for x in b], [x[1] for x in b], truncation="only_first",
                  max_length=256, padding=True, return_tensors="pt").to(device)
        y = torch.tensor([x[2] for x in b]).to(device)
        loss = model(**enc, labels=y).loss
        if torch.isnan(loss):
            raise RuntimeError(f"nan loss at epoch {ep+1} step {step}")
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); opt.zero_grad()
        total += loss.item(); n += 1
        if step % 20 == 0:
            print(f"epoch {ep+1} step {step}/{len(data)//BS} loss {loss.item():.3f} | {time.time()-t0:.0f}s", flush=True)
    print("=== epoch", ep + 1, "avg loss", round(total / n, 4))
m

epoch 1 step 0/78 loss 1.968 | 1s
epoch 1 step 20/78 loss 0.541 | 13s
epoch 1 step 40/78 loss 0.822 | 25s
epoch 1 step 60/78 loss 0.601 | 38s
=== epoch 1 avg loss 0.6705
epoch 2 step 0/78 loss 0.287 | 51s
epoch 2 step 20/78 loss 0.181 | 64s
epoch 2 step 40/78 loss 0.383 | 77s
epoch 2 step 60/78 loss 0.151 | 90s
=== epoch 2 avg loss 0.2332
epoch 3 step 0/78 loss 0.015 | 102s
epoch 3 step 20/78 loss 0.047 | 115s
epoch 3 step 40/78 loss 0.188 | 128s
epoch 3 step 60/78 loss 0.008 | 141s
=== epoch 3 avg loss 0.1027


NameError: name 'm' is not defined

In [71]:
model.eval();

In [72]:
model.eval()
for topk in [1, 2, 3]:
    p = run([(ce_input(i, topk), df.claim.iloc[i]) for i in range(len(df))])
    print("top-k =", topk, "| acc:", round((df.gold.values == np.array(p)).mean(), 3))
    print(classification_report(df.gold, p, digits=3, zero_division=0))

top-k = 1 | acc: 0.684
              precision    recall  f1-score   support

  CONTRADICT      0.690     0.730     0.709       122
      NOINFO      0.638     0.268     0.377       112
     SUPPORT      0.690     0.875     0.771       216

    accuracy                          0.684       450
   macro avg      0.673     0.624     0.619       450
weighted avg      0.677     0.684     0.656       450

top-k = 2 | acc: 0.698
              precision    recall  f1-score   support

  CONTRADICT      0.729     0.639     0.681       122
      NOINFO      0.509     0.741     0.604       112
     SUPPORT      0.850     0.708     0.773       216

    accuracy                          0.698       450
   macro avg      0.696     0.696     0.686       450
weighted avg      0.732     0.698     0.706       450

top-k = 3 | acc: 0.636
              precision    recall  f1-score   support

  CONTRADICT      0.889     0.525     0.660       122
      NOINFO      0.422     0.946     0.584       112
     S

In [73]:
ce_scores = [ce.predict([(c, s) for s in sent_map[d]], show_progress_bar=False)
             for c, d in zip(df.claim, df.doc_id)]

top1 = np.array([s.max() for s in ce_scores])
for lab in ["SUPPORT", "CONTRADICT", "NOINFO"]:
    v = top1[df.gold.values == lab]
    print(lab, "| top-1 CE score: mean", round(v.mean(), 2),
          "| median", round(np.median(v), 2))

SUPPORT | top-1 CE score: mean 4.35 | median 4.79
CONTRADICT | top-1 CE score: mean 1.96 | median 1.83
NOINFO | top-1 CE score: mean -2.84 | median -2.98


In [74]:
from sklearn.metrics import roc_auc_score, f1_score

is_noinfo = df.gold.values == "NOINFO"
print("AUC (has evidence vs NOINFO):", round(roc_auc_score(~is_noinfo, top1), 3))

ids = df.id.unique(); rng = np.random.RandomState(0); rng.shuffle(ids)
A = set(ids[:len(ids)//2])
idx_A = [i for i in range(len(df)) if df.id.iloc[i] in A]
idx_B = [i for i in range(len(df)) if df.id.iloc[i] not in A]

def predict_thr(t, idxs, maxk=2):
    out, inputs, keep = {}, [], []
    for i in idxs:
        s = ce_scores[i]
        sel = [j for j in np.argsort(-s)[:maxk] if s[j] > t]
        if not sel:
            out[i] = "NOINFO"
        else:
            inputs.append((" ".join(sent_map[df.doc_id.iloc[i]][j] for j in sorted(sel)), df.claim.iloc[i]))
            keep.append(i)
    if inputs:
        for i, p in zip(keep, run(inputs)):
            out[i] = p
    return [out[i] for i in idxs]

def mf1(idxs, preds):
    return f1_score(df.gold.values[idxs], preds, average="macro")

best = None
for t in [-4, -3, -2, -1, 0, 1, 2]:
    f = mf1(idx_A, predict_thr(t, idx_A))
    print("t =", t, "| half A macro-F1:", round(f, 3))
    if best is None or f > best[1]:
        best = (t, f)
print("chosen t:", best[0])

pB = predict_thr(best[0], idx_B)
print("Half B | threshold:", round(mf1(idx_B, pB), 3),
      "| acc:", round((df.gold.values[idx_B] == np.array(pB)).mean(), 3))

p2 = run([(ce_input(i, 2), df.claim.iloc[i]) for i in idx_B])
print("Half B | fixed top-2:", round(mf1(idx_B, p2), 3),
      "| acc:", round((df.gold.values[idx_B] == np.array(p2)).mean(), 3))

AUC (has evidence vs NOINFO): 0.893
t = -4 | half A macro-F1: 0.686
t = -3 | half A macro-F1: 0.701
t = -2 | half A macro-F1: 0.689
t = -1 | half A macro-F1: 0.678
t = 0 | half A macro-F1: 0.66
t = 1 | half A macro-F1: 0.667
t = 2 | half A macro-F1: 0.611
chosen t: -3
Half B | threshold: 0.716 | acc: 0.737
Half B | fixed top-2: 0.67 | acc: 0.696
